In [5]:
!pip install -q requests beautifulsoup4 pandas tqdm nltk

In [6]:
import os

folders = [
    "VictorianGPT",
    "VictorianGPT/raw",
    "VictorianGPT/cleaned",
    "VictorianGPT/dialogues"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created")

Folders created


In [7]:
import requests

books = {

"great_expectations":
"https://www.gutenberg.org/cache/epub/1400/pg1400.txt",

"jane_eyre":
"https://www.gutenberg.org/cache/epub/1260/pg1260.txt",

"dracula":
"https://www.gutenberg.org/cache/epub/345/pg345.txt",

"dorian_gray":
"https://www.gutenberg.org/cache/epub/174/pg174.txt"

}

for name,url in books.items():

    text=requests.get(url).text

    with open(
        f"VictorianGPT/raw/{name}.txt",
        "w",
        encoding="utf8"
    ) as f:

        f.write(text)

    print(name,"downloaded")

great_expectations downloaded
jane_eyre downloaded
dracula downloaded
dorian_gray downloaded


In [8]:
import re
import os

raw="VictorianGPT/raw"
clean="VictorianGPT/cleaned"

for file in os.listdir(raw):

    with open(
        f"{raw}/{file}",
        encoding="utf8"
    ) as f:

        text=f.read()

    start="*** START"
    end="*** END"

    s=text.find(start)
    e=text.find(end)

    if s!=-1 and e!=-1:
        text=text[s:e]

    text=re.sub(r"\n+","\n",text)
    text=re.sub(r"\s+"," ",text)

    with open(
        f"{clean}/{file}",
        "w",
        encoding="utf8"
    ) as f:

        f.write(text)

print("cleaning done")

cleaning done


In [12]:
import nltk
from nltk.tokenize import sent_tokenize
import os

# Ensure NLTK data path is set and exists for robustness in Colab
nltk_data_dir = '/root/nltk_data'
if nltk_data_dir not in nltk.data.path:
    nltk.data.path.append(nltk_data_dir)
os.makedirs(nltk_data_dir, exist_ok=True)

# Check if 'punkt' is available. If not, download it.
try:
    nltk.data.find('tokenizers/punkt')
    print("'punkt' NLTK resource found.")
except LookupError:
    print("'punkt' NLTK resource not found, attempting download...")
    nltk.download('punkt', download_dir=nltk_data_dir)
    try:
        nltk.data.find('tokenizers/punkt')
        print("'punkt' NLTK resource downloaded successfully.")
    except LookupError:
        print("Error: 'punkt' NLTK resource still not found after download.")

# Check if 'punkt_tab' (a dependency for PunktTokenizer) is available. If not, download it.
try:
    nltk.data.find('tokenizers/punkt_tab')
    print("'punkt_tab' NLTK resource found.")
except LookupError:
    print("'punkt_tab' NLTK resource not found, attempting download...")
    nltk.download('punkt_tab', download_dir=nltk_data_dir)
    try:
        nltk.data.find('tokenizers/punkt_tab')
        print("'punkt_tab' NLTK resource downloaded successfully.")
    except LookupError:
        print("Error: 'punkt_tab' NLTK resource still not found after download. There might be a persistent issue with NLTK data installation.")

# Assume 'clean' is defined from previous cells. Explicitly redefine for robustness.
clean = "VictorianGPT/cleaned"

chunks = []

if not os.path.exists(clean):
    print(f"Error: Directory '{clean}' not found. Please ensure previous steps created and populated it.")
else:
    for file_name in os.listdir(clean):
        file_path = os.path.join(clean, file_name)
        if os.path.isfile(file_path): # Process only files, not directories
            with open(file_path, "r", encoding="utf8") as f:
                text = f.read()

            if text.strip(): # Avoid processing empty or whitespace-only strings
                sentences = sent_tokenize(text)
                for i in range(0, len(sentences), 8):
                    block = " ".join(sentences[i:i+8])
                    chunks.append(block)
            else:
                print(f"Warning: Skipping empty or whitespace-only file: {file_name}")

    print(f"Total chunks created: {len(chunks)}")

'punkt' NLTK resource found.
'punkt_tab' NLTK resource not found, attempting download...


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


'punkt_tab' NLTK resource downloaded successfully.
Total chunks created: 3359


In [13]:
import pandas as pd

df=pd.DataFrame({
    "text":chunks
})

df.to_csv(
    "VictorianGPT/chunks.csv",
    index=False)

print(df.head())

                                                text
0  *** START OF THE PROJECT GUTENBERG EBOOK JANE ...
1  I would suggest to such doubters certain obvio...
2  The world may not like to see these ideas diss...
3  They say he is like Fielding: they talk of his...
4  CURRER BELL. _April_ 13_th_, 1848. CHAPTER I T...


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import shutil

source="/content/VictorianGPT"
destination="/content/drive/MyDrive/VictorianGPT"

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

print("VictorianGPT copied to Google Drive")

VictorianGPT copied to Google Drive


In [18]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/VictorianGPT"):
    print(root)
    print(files)

/content/drive/MyDrive/VictorianGPT
['chunks.csv']
/content/drive/MyDrive/VictorianGPT/raw
['jane_eyre.txt', 'great_expectations.txt', 'dorian_gray.txt', 'dracula.txt']
/content/drive/MyDrive/VictorianGPT/dialogues
[]
/content/drive/MyDrive/VictorianGPT/cleaned
['jane_eyre.txt', 'great_expectations.txt', 'dorian_gray.txt', 'dracula.txt']
